# Implementation 1: The SOR Method (Algorithm 7.3)

**Math 455 Midterm Project: Relaxation Methods**

This notebook implements Algorithm 7.3 (Successive Over-Relaxation) from Burden, Faires & Burden,
*Numerical Analysis*, 10th ed., p. 473, and tests it against Example 1 on p. 471.

In [1]:
from collections import namedtuple

import numpy as np

## Algorithm 7.3

Each sweep updates the components in order, using new values for $j < i$ and old values for $j > i$:

$$x_i = (1-\omega)XO_i + \frac{1}{a_{ii}}\left[\omega\left(-\sum_{j=1}^{i-1} a_{ij}x_j - \sum_{j=i+1}^{n} a_{ij}XO_j + b_i\right)\right]$$

Iteration stops when $\|\mathbf{x} - \mathbf{XO}\| < TOL$. The book does not name a norm in the algorithm, so
`sor` defaults to the $l_\infty$ norm used in Exercise Set 7.4. Setting $\omega = 1$ gives Gauss-Seidel.

The code comments mark the steps of the algorithm. Three things are added to the book's version:
the check for $a_{ii} \neq 0$, which the method requires; the iterate history, which the tests need; and
returning the last iterate on failure, where the book only prints a message.

In [2]:
SORResult = namedtuple("SORResult", ["x", "iterations", "converged", "history"])


def sor(A, b, x0, omega, tol, max_iter, norm=np.inf):
    '''Solve Ax = b by SOR (Burden, Faires & Burden, Algorithm 7.3).

    Returns SORResult(x, iterations, converged, history), where history[k] is
    the iterate x^(k) and history[0] is x0.
    '''
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    XO = np.array(x0, dtype=float)
    n = len(b)

    if np.any(np.diag(A) == 0):
        raise ValueError("SOR requires every diagonal entry a_ii to be nonzero")

    x = np.zeros(n)
    history = [XO.copy()]

    k = 1                                                     # Step 1
    while k <= max_iter:                                      # Step 2
        for i in range(n):                                    # Step 3
            x[i] = (1 - omega) * XO[i] + (1 / A[i, i]) * (
                omega * (-A[i, :i] @ x[:i] - A[i, i + 1:] @ XO[i + 1:] + b[i])
            )
        history.append(x.copy())

        if np.linalg.norm(x - XO, norm) < tol:                # Step 4
            return SORResult(x.copy(), k, True, history)

        k += 1                                                # Step 5
        XO = x.copy()                                         # Step 6

    print("Maximum number of iterations exceeded")            # Step 7
    return SORResult(x.copy(), max_iter, False, history)

## Example 1 (p. 471)

$$\begin{aligned}
4x_1 + 3x_2 \phantom{{}-x_3} &= 24\\
3x_1 + 4x_2 - x_3 &= 30\\
-x_2 + 4x_3 &= -24
\end{aligned}$$

The exact solution is $(3, 4, -5)^t$, and both methods start from $\mathbf{x}^{(0)} = (1, 1, 1)^t$. The arrays
below hold the book's published iterates: `TABLE_7_3` for Gauss-Seidel and `TABLE_7_4` for SOR with
$\omega = 1.25$. Each row is one iterate $k = 0, \dots, 7$.

In [3]:
A = np.array([[4, 3, 0],
              [3, 4, -1],
              [0, -1, 4]])
b = np.array([24, 30, -24])
x0 = np.array([1, 1, 1])
exact = np.array([3, 4, -5])

# Transposed so that row k holds (x1, x2, x3) at iteration k.
TABLE_7_3 = np.array([
    [1, 5.2500000, 3.1406250, 3.0878906, 3.0549316, 3.0343323, 3.0214577, 3.0134110],
    [1, 3.8125000, 3.8828125, 3.9267578, 3.9542236, 3.9713898, 3.9821186, 3.9888241],
    [1, -5.0468750, -5.0292969, -5.0183105, -5.0114441, -5.0071526, -5.0044703, -5.0027940],
]).T

TABLE_7_4 = np.array([
    [1, 6.3125000, 2.6223145, 3.1333027, 2.9570512, 3.0037211, 2.9963276, 3.0000498],
    [1, 3.5195313, 3.9585266, 4.0102646, 4.0074838, 4.0029250, 4.0009262, 4.0002586],
    [1, -6.6501465, -4.6004238, -5.0966863, -4.9734897, -5.0057135, -4.9982822, -5.0003486],
]).T

# A tight tolerance keeps the run going well past the iterates being compared.
gs = sor(A, b, x0, omega=1.0, tol=1e-12, max_iter=200)
sr = sor(A, b, x0, omega=1.25, tol=1e-12, max_iter=200)

### Test 1: Reproduce Tables 7.3 and 7.4

The book prints 7 decimal places, so a correct iterate must agree with its table entry to within
rounding error, $0.5 \times 10^{-7}$. The small extra allowance in `ROUNDING` covers floating-point
error in the comparison itself.

In [4]:
ROUNDING = 0.5e-7 + 1e-12


def show_table(title, computed, book):
    print(title)
    print(f"{'k':>2} {'x1':>12} {'x2':>12} {'x3':>12} {'max |computed - book|':>24}")
    for k, row in enumerate(computed):
        print(f"{k:>2} {row[0]:>12.7f} {row[1]:>12.7f} {row[2]:>12.7f} "
              f"{np.max(np.abs(row - book[k])):>24.2e}")
    print()


for title, result, book in [("Table 7.3: Gauss-Seidel (omega = 1)", gs, TABLE_7_3),
                            ("Table 7.4: SOR (omega = 1.25)", sr, TABLE_7_4)]:
    computed = np.array(result.history[:8])
    show_table(title, computed, book)
    assert np.all(np.abs(computed - book) <= ROUNDING), f"{title} does not match the book"

print("PASS: all iterates match Tables 7.3 and 7.4 to the book's 7 decimal places")

Table 7.3: Gauss-Seidel (omega = 1)
 k           x1           x2           x3    max |computed - book|
 0    1.0000000    1.0000000    1.0000000                 0.00e+00
 1    5.2500000    3.8125000   -5.0468750                 0.00e+00
 2    3.1406250    3.8828125   -5.0292969                 2.50e-08
 3    3.0878906    3.9267578   -5.0183105                 4.69e-08
 4    3.0549316    3.9542236   -5.0114441                 4.06e-08
 5    3.0343323    3.9713898   -5.0071526                 4.26e-08
 6    3.0214577    3.9821186   -5.0044703                 4.84e-08
 7    3.0134110    3.9888241   -5.0027940                 4.51e-08

Table 7.4: SOR (omega = 1.25)
 k           x1           x2           x3    max |computed - book|
 0    1.0000000    1.0000000    1.0000000                 0.00e+00
 1    6.3125000    3.5195312   -6.6501465                 5.00e-08
 2    2.6223145    3.9585266   -4.6004238                 4.69e-08
 3    3.1333027    4.0102646   -5.0966863                 4.83

In Table 7.4, $x_2^{(1)}$ is exactly $3.51953125$, a tie at the 7th decimal. The book rounds it up to
$3.5195313$, while Python's formatting rounds it to the even digit and prints $3.5195312$. The value itself
is the same, which is why this row shows a difference of exactly $5 \times 10^{-8}$.

### Test 2: Agreement with the example's explicit equations

Example 1 also writes out each method's update equations with numeric coefficients. Coding these directly
gives a check at full double precision, rather than only to the table's 7 decimal places.

In [5]:
def gs_step(x):
    x1 = -0.75 * x[1] + 6
    x2 = -0.75 * x1 + 0.25 * x[2] + 7.5
    x3 = 0.25 * x2 - 6
    return np.array([x1, x2, x3])


def sor_step(x):
    x1 = -0.25 * x[0] - 0.9375 * x[1] + 7.5
    x2 = -0.9375 * x1 - 0.25 * x[1] + 0.3125 * x[2] + 9.375
    x3 = 0.3125 * x2 - 0.25 * x[2] - 7.5
    return np.array([x1, x2, x3])


for name, result, step in [("Gauss-Seidel", gs, gs_step), ("SOR", sr, sor_step)]:
    x = x0.astype(float)
    for k in range(1, 21):
        x = step(x)
        assert np.max(np.abs(x - result.history[k])) < 1e-12, f"{name} differs at k = {k}"

print("PASS: 20 iterates of each method match the explicit equations to within 1e-12")

PASS: 20 iterates of each method match the explicit equations to within 1e-12


### Test 3: Iteration counts (34 vs. 14)

The book states that being "accurate to seven decimal places" takes 34 Gauss-Seidel iterations but only 14
SOR iterations. Reading this as a true error below $0.5 \times 10^{-7}$ reproduces both counts.

In [6]:
def first_accurate(result, exact, threshold=0.5e-7):
    '''First k whose true error ||x^(k) - exact||_inf is below threshold.'''
    for k, x in enumerate(result.history):
        if np.linalg.norm(x - exact, np.inf) < threshold:
            return k
    return None


gs_count = first_accurate(gs, exact)
sr_count = first_accurate(sr, exact)
print(f"Gauss-Seidel: {gs_count} iterations (book: 34)")
print(f"SOR:          {sr_count} iterations (book: 14)")

assert gs_count == 34 and sr_count == 14
print("PASS: iteration counts match the book")

Gauss-Seidel: 34 iterations (book: 34)
SOR:          14 iterations (book: 14)
PASS: iteration counts match the book


### Test 4: Algorithm 7.3's own stopping rule

Test 3 measures the true error, which is only available because the exact solution is known. Algorithm 7.3
instead stops on the change between iterates, $\|\mathbf{x} - \mathbf{XO}\|_\infty < TOL$, so the iteration
counts it reports need not equal 34 and 14. This test runs the algorithm as written and checks the answer
it returns.

In [7]:
TOL = 1e-7
print(f"TOL = {TOL:g}, l_inf norm")
print(f"{'method':<14} {'omega':>6} {'iterations':>11} {'true error':>12}")
for name, omega in [("Gauss-Seidel", 1.0), ("SOR", 1.25)]:
    result = sor(A, b, x0, omega=omega, tol=TOL, max_iter=100)
    error = np.linalg.norm(result.x - exact, np.inf)
    print(f"{name:<14} {omega:>6} {result.iterations:>11} {error:>12.2e}")
    assert result.converged and error < 10 * TOL

print("PASS: both runs converge to (3, 4, -5) under the algorithm's stopping rule")

TOL = 1e-07, l_inf norm
method          omega  iterations   true error
Gauss-Seidel      1.0          32     1.06e-07
SOR              1.25          15     1.19e-08
PASS: both runs converge to (3, 4, -5) under the algorithm's stopping rule


With this tolerance the algorithm stops after 32 Gauss-Seidel iterations and 15 SOR iterations, and no single
$TOL$ gives both 34 and 14. Gauss-Seidel stops at $k = 34$ only for $TOL$ between about $2.5 \times 10^{-8}$
and $4.0 \times 10^{-8}$, while SOR stops at $k = 14$ only for $TOL$ between about $1.4 \times 10^{-7}$ and
$8.1 \times 10^{-7}$. The comparison still favours SOR: it needs less than half the iterations.

### Test 5: Failure handling

Step 7 must report failure when $N$ iterations are not enough, and a zero on the diagonal must be rejected
because Step 3 divides by $a_{ii}$.

In [8]:
result = sor(A, b, x0, omega=1.25, tol=1e-7, max_iter=5)
assert not result.converged and result.iterations == 5 and len(result.history) == 6

try:
    sor([[0, 1], [1, 0]], [1, 1], [0, 0], omega=1.25, tol=1e-7, max_iter=5)
except ValueError as e:
    print(f"Rejected: {e}")
else:
    raise AssertionError("a zero diagonal entry should raise ValueError")

print("PASS: failure cases are reported")

Maximum number of iterations exceeded
Rejected: SOR requires every diagonal entry a_ii to be nonzero
PASS: failure cases are reported
